In [0]:
# Replace with your actual storage account name and key
spark.conf.set(
"fs.azure.account.key.projectlinkedinjobs.dfs.core.windows.net",
"<AccountKey>"
)

In [0]:
gold = spark.read.format("delta").load(
    "abfss://lakehouse@projectlinkedinjobs.dfs.core.windows.net/gold/curated_postings/"
)
gold.show(5, truncate=False)
print("Rows:", gold.count())

+------------------------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------+--------------------------+-------------------+-----------+--------------+------------------------+----------+--------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------

In [0]:
# check the data types
gold.printSchema()

root
 |-- job_link: string (nullable = true)
 |-- job_title: string (nullable = true)
 |-- company: string (nullable = true)
 |-- job_location: string (nullable = true)
 |-- search_city: string (nullable = true)
 |-- search_country: string (nullable = true)
 |-- search_position: string (nullable = true)
 |-- job_level: string (nullable = true)
 |-- job_type: string (nullable = true)
 |-- job_skills: string (nullable = true)
 |-- job_summary: string (nullable = true)



In [0]:
from pyspark.sql import functions as F

df = gold  


In [0]:
# 1) lowercase + trim
df = df.withColumn("job_title_clean", F.lower(F.trim(F.col("job_title"))))

# 2) remove seniority / level words
df = df.withColumn(
    "job_title_clean",
    F.regexp_replace(
        "job_title_clean",
        r"\b(senior|sr\.?|junior|jr\.?|lead|principal|staff|intern|trainee)\b",
        ""
    )
)

# 3) remove roman numerals like II, III, IV
df = df.withColumn(
    "job_title_clean",
    F.regexp_replace("job_title_clean", r"\b(i|ii|iii|iv|v)\b", "")
)

# 4) remove non-letters
df = df.withColumn(
    "job_title_clean",
    F.regexp_replace("job_title_clean", r"[^a-z\s]", " ")
)

# 5) collapse multiple spaces, trim again
df = df.withColumn(
    "job_title_clean",
    F.regexp_replace("job_title_clean", r"\s+", " ")
)
df = df.withColumn("job_title_clean", F.trim(F.col("job_title_clean")))

# 6) drop empty titles
df = df.filter(F.col("job_title_clean") != "")


In [0]:
df.select("job_title_clean").distinct().count()


493796

In [0]:
top_n = 100

top_roles_df = (
    df.groupBy("job_title_clean")
      .count()
      .orderBy(F.col("count").desc())
      .limit(top_n)
)

top_roles = [r["job_title_clean"] for r in top_roles_df.collect()]

# Optional: see them
display(top_roles_df.orderBy(F.col("count").desc()))


job_title_clean,count
customer service representative,10010
sales associate ft,7325
store manager,6314
shift manager,6028
accountant,5899
assistant manager,5729
first year tax professional,5356
sales associate pt,4924
registered nurse,4640
account executive,3085


In [0]:
df_roles = df.filter(F.col("job_title_clean").isin(top_roles)) \
             .withColumn("job_role", F.col("job_title_clean"))


In [0]:
from pyspark.ml.feature import StringIndexer

label_indexer = StringIndexer(
    inputCol="job_role",
    outputCol="label",
    handleInvalid="skip"
)

label_model = label_indexer.fit(df_roles)
df_labeled = label_model.transform(df_roles)


In [0]:
df_labeled.select("label").show()

+-----+
|label|
+-----+
|  2.0|
|  0.0|
| 18.0|
|  1.0|
|  1.0|
| 71.0|
|  1.0|
|  0.0|
|  4.0|
| 60.0|
| 36.0|
| 17.0|
| 27.0|
|  0.0|
|  9.0|
| 60.0|
|  1.0|
| 17.0|
| 51.0|
|  2.0|
+-----+
only showing top 20 rows


In [0]:
df_labeled = df_labeled.drop("search_position")


In [0]:
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.stat import ChiSquareTest

# Index search_country
indexer_country = StringIndexer(
    inputCol="search_country",
    outputCol="search_country_idx",
    handleInvalid="keep"
).fit(df_labeled)

df_country_i = indexer_country.transform(df_labeled)

# Assemble into features vector
assembler_country = VectorAssembler(
    inputCols=["search_country_idx"],
    outputCol="features"
)

df_country_f = assembler_country.transform(df_country_i)

# Run Chi-square
chi_df = ChiSquareTest.test(df_country_f, "features", "label")

chi_df.show(truncate=False)   # just to see the raw output

row = chi_df.head()

print("=== search_country ===")
print("p-values:", row.pValues)
print("statistics:", row.statistics)
print("degreesOfFreedom:", row.degreesOfFreedom)

# Since we only have 1 feature, you can take index 0:
p_value = float(row.pValues[0])
statistic = float(row.statistics[0])
dfreedom = int(row.degreesOfFreedom[0])

print("\nSingle feature result:")
print("p-value:", p_value)
print("statistic:", statistic)
print("df:", dfreedom)


+-------+----------------+-------------------+
|pValues|degreesOfFreedom|statistics         |
+-------+----------------+-------------------+
|[0.0]  |[297]           |[44300.83927958655]|
+-------+----------------+-------------------+

=== search_country ===
p-values: [0.0]
statistics: [44300.83927958655]
degreesOfFreedom: [297]

Single feature result:
p-value: 0.0
statistic: 44300.83927958655
df: 297


In [0]:
df_labeled = df_labeled.drop("job_location")

In [0]:
column_names = df_labeled.columns
print(column_names)

['job_link', 'job_title', 'company', 'search_city', 'search_country', 'job_level', 'job_type', 'job_skills', 'job_summary', 'job_title_clean', 'job_role', 'label']


In [0]:
feature_cols = [
    "company",
    "search_country",
    "job_level",
    "job_type",
    "job_skills",
    "job_summary",
    "job_role",   
    "label"       
]

df_features = df_labeled.select(*feature_cols)


In [0]:
from pyspark.sql.functions import col, lower, trim, regexp_replace, when

# categorical columns
cat_cols = ["company", "search_country", "job_level", "job_type"]

for c in cat_cols:
    df_features = df_features.withColumn(c, lower(trim(col(c))))
    df_features = df_features.withColumn(c, regexp_replace(col(c), r"\s+", " "))
    df_features = df_features.withColumn(c, when(col(c).isNull(), "unknown").otherwise(col(c)))

# text columns
text_cols = ["job_skills", "job_summary"]

for c in text_cols:
    df_features = df_features.withColumn(c, lower(col(c)))
    df_features = df_features.withColumn(c, regexp_replace(col(c), r"\n", " "))
    df_features = df_features.withColumn(c, regexp_replace(col(c), r"\s+", " "))
    df_features = df_features.withColumn(c, trim(col(c)))


In [0]:
df_features.groupBy("company").count().orderBy("count", ascending=False).show(20)


+--------------------+-----+
|             company|count|
+--------------------+-----+
|      dollar general|14073|
|          mcdonald's| 6405|
|           h&r block| 5359|
|             walmart| 5274|
|       family dollar| 4289|
|              target| 3435|
|   travelnursesource| 3227|
|            circle k| 2720|
|  dollar tree stores| 2296|
|        sally beauty| 2235|
|               jobot| 2220|
|gotham enterprise...| 2167|
|   jobs for humanity| 2094|
|            gamestop| 2032|
|     texas roadhouse| 1859|
|bob evans restaur...| 1858|
|      the home depot| 1674|
|        olive garden| 1494|
|   ross stores, inc.| 1482|
|         robert half| 1245|
+--------------------+-----+
only showing top 20 rows


In [0]:
display(df_features.groupBy("company").count().orderBy("count", ascending=False).limit(20))


company,count
dollar general,14073
mcdonald's,6405
h&r block,5359
walmart,5274
family dollar,4289
target,3435
travelnursesource,3227
circle k,2720
dollar tree stores,2296
sally beauty,2235


In [0]:
display(df_features.groupBy("job_type").count())


job_type,count
onsite,183279
hybrid,1832
remote,561


In [0]:
display(df_features.groupBy("job_level").count())


job_level,count
associate,14440
mid senior,171232


In [0]:
from pyspark.sql.functions import col

display(
    df_features.groupBy("search_country")
               .count()
               .orderBy(col("count").desc())
)


search_country,count
united states,167425
united kingdom,11338
canada,4342
australia,2567


In [0]:
from pyspark.sql.functions import length

display(df_features.select(length("job_skills").alias("skills_length")))


skills_length
216
360
264
305
712
659
435
445
204
387


In [0]:
display(df_features.select(length("job_summary").alias("summary_length")))


summary_length
1383
2976
3686
null
null
3895
null
2984
1402
null


In [0]:
display(df_features.groupBy("job_role").count().orderBy("count", ascending=False).limit(30))


job_role,count
customer service representative,10010
sales associate ft,7325
store manager,6314
shift manager,6028
accountant,5899
assistant manager,5729
first year tax professional,5356
sales associate pt,4924
registered nurse,4640
account executive,3085


In [0]:
from pyspark.sql.functions import col, lower, trim, regexp_replace, when

df_features_clean = df_features

# ------------------------
# A. CLEAN CATEGORICAL COLUMNS
# ------------------------

cat_cols = ["company", "search_country", "job_level", "job_type"]

for c in cat_cols:
    df_features_clean = df_features_clean.withColumn(c, lower(trim(col(c))))
    df_features_clean = df_features_clean.withColumn(c, regexp_replace(col(c), r"\s+", " "))
    df_features_clean = df_features_clean.withColumn(c, when(col(c).isNull() | (col(c) == ""), "unknown").otherwise(col(c)))


# ------------------------
# B. CLEAN TEXT COLUMNS
# ------------------------

text_cols = ["job_skills", "job_summary"]

for c in text_cols:
    df_features_clean = df_features_clean.withColumn(c, lower(col(c)))
    df_features_clean = df_features_clean.withColumn(c, regexp_replace(col(c), r"\n", " "))
    df_features_clean = df_features_clean.withColumn(c, regexp_replace(col(c), r"\s+", " "))
    df_features_clean = df_features_clean.withColumn(c, trim(col(c)))
    df_features_clean = df_features_clean.withColumn(c, when(col(c).isNull() | (col(c) == ""), "no_information").otherwise(col(c)))


# ------------------------
# C. ENSURE TARGET COLUMNS HAVE NO NULLS
# ------------------------

df_features_clean = df_features_clean.withColumn(
    "job_role", when(col("job_role").isNull() | (col("job_role") == ""), "unknown_role").otherwise(col("job_role"))
)

df_features_clean = df_features_clean.withColumn(
    "label", when(col("label").isNull(), -1).otherwise(col("label"))
)



In [0]:
from pyspark.sql.functions import sum

null_counts = df_features_clean.select(
    *[sum(col(c).isNull().cast("int")).alias(c) for c in df_features_clean.columns]
)

display(null_counts)


company,search_country,job_level,job_type,job_skills,job_summary,job_role,label
0,0,0,0,0,0,0,0


In [0]:
from pyspark.sql.functions import col

display(
    df_features_clean.groupBy("company")
                     .count()
                     .orderBy(col("count").desc())
                     .limit(20)
)


company,count
dollar general,14073
mcdonald's,6405
h&r block,5359
walmart,5274
family dollar,4289
target,3435
travelnursesource,3227
circle k,2720
dollar tree stores,2296
sally beauty,2235


In [0]:
display(
    df_features_clean.groupBy("search_country")
                     .count()
                     .orderBy(col("count").desc())
)


search_country,count
united states,167425
united kingdom,11338
canada,4342
australia,2567


In [0]:
display(
    df_features_clean.groupBy("job_level")
                     .count()
                     .orderBy(col("count").desc())
)


job_level,count
mid senior,171232
associate,14440


In [0]:
display(
    df_features_clean.groupBy("job_type")
                     .count()
                     .orderBy(col("count").desc())
)


job_type,count
onsite,183279
hybrid,1832
remote,561


In [0]:
display(
    df_features_clean.groupBy("job_role")
                     .count()
                     .orderBy(col("count").desc())
                     .limit(30)
)


job_role,count
customer service representative,10010
sales associate ft,7325
store manager,6314
shift manager,6028
accountant,5899
assistant manager,5729
first year tax professional,5356
sales associate pt,4924
registered nurse,4640
account executive,3085


Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.functions import length

display(
    df_features_clean.withColumn("summary_len", length("job_summary"))
)


company,search_country,job_level,job_type,job_skills,job_summary,job_role,label,summary_len
rhr,united kingdom,mid senior,onsite,"store management, retail management, commercial awareness, leadership, sales drive, training and development, customer service, team management, profitability maximization, cost containment, fashion retail experience","here at edinburgh woollen mill we have a fantastic opportunity for a store manager based in our stratford store. if you would like to be part of a forward,hinking business and have a job with excellent career prospects, we would love to hear from you. as well as being part of an exciting and dynamic team you will also have the opportunity to gain an industry recognised qualification within your first 18 months of you wish to do so. as store manager we are looking for a well rounded and commercial ‘retail manager’ who is proactive in their approach and can work using their own initiative, fashion retail experience desirable however not essential as full training provided. the ideal candidate will have previous experience either in management or supervision of personnel within a retail outlet. as store manager, you will be expected to drive sales through your team manage the day to day operation of the store whilst ensuring costs are contained within targets. maximize store profitability by promoting sales within the store. ensure that a high level of customer service is delivered at all times. manage, coach and motivate the team to deliver to all targets and lead by example. the ideal candidate will have commercial awareness excellent leadership credentials an ability to drive sales through your team good training and development capabilities show more show less",store manager,2.0,1383
circle k,united states,mid senior,onsite,"customer service, retail experience, cashiering experience, communication skills, teamwork, problemsolving skills, high school diploma or equivalent, ability to stand for 8 hours, ability to lift and carry up to 60 pounds, ability to push,ull with arms up to 20 pounds, ability to bend at the waist and grasp objects, eyehand coordination, willingness to learn","store 2723836: 1008 s main st, graysville, alabama 35073 availability ,shift,ays flexible availability customer service representative we want you to join our team as a customer service representative. if you have the desire to be challenged, work in a fast,aced, fun environment and to grow your career ,look no further. as a customer service representative, you will enjoy medical, vision, dental, & life insurance,hort & long term disability flexible schedules weekly pay full,ime or part,ime large, stable employer fast career opportunities work with fun, motivated people task variety paid comprehensive training 401k with a competitive company match flexible spending,ealth savings accounts tuition reimbursement your key responsibilities you will greet customers, run the register, cashier, make purchase suggestions and sometimes work with our food program. there is never a dull moment as you will be working around the store (inside and out) in many different areas to help maintain our high standards for store appearance and provide fast and friendly service to our customers. provide regular and predicable onsite attendance. you will interact with many customers daily, all while working with a fun, energetic team accomplishing daily tasks around the store! you are good at selling products to customers providing excellent customer care communication and friendly conversation performing at a quick pace while having fun working as part of a team to accomplish daily goals coming up with great ideas to solve problems thinking quickly and offering suggestions great if you have retail and customer service experience sales associate or cashiering experience high school diploma or equivalent motivation to advance in your career! willingness to learn and have fun! physical requirements ability to stand and,r walk for up 

In [0]:
display(
    df_features_clean.withColumn("skills_len", length("job_skills"))
)
df = gold.withColumn("job_title_clean", lower(trim(col("job_title"))))


company,search_country,job_level,job_type,job_skills,job_summary,job_role,label,skills_len
rhr,united kingdom,mid senior,onsite,"store management, retail management, commercial awareness, leadership, sales drive, training and development, customer service, team management, profitability maximization, cost containment, fashion retail experience","here at edinburgh woollen mill we have a fantastic opportunity for a store manager based in our stratford store. if you would like to be part of a forward,hinking business and have a job with excellent career prospects, we would love to hear from you. as well as being part of an exciting and dynamic team you will also have the opportunity to gain an industry recognised qualification within your first 18 months of you wish to do so. as store manager we are looking for a well rounded and commercial ‘retail manager’ who is proactive in their approach and can work using their own initiative, fashion retail experience desirable however not essential as full training provided. the ideal candidate will have previous experience either in management or supervision of personnel within a retail outlet. as store manager, you will be expected to drive sales through your team manage the day to day operation of the store whilst ensuring costs are contained within targets. maximize store profitability by promoting sales within the store. ensure that a high level of customer service is delivered at all times. manage, coach and motivate the team to deliver to all targets and lead by example. the ideal candidate will have commercial awareness excellent leadership credentials an ability to drive sales through your team good training and development capabilities show more show less",store manager,2.0,216
circle k,united states,mid senior,onsite,"customer service, retail experience, cashiering experience, communication skills, teamwork, problemsolving skills, high school diploma or equivalent, ability to stand for 8 hours, ability to lift and carry up to 60 pounds, ability to push,ull with arms up to 20 pounds, ability to bend at the waist and grasp objects, eyehand coordination, willingness to learn","store 2723836: 1008 s main st, graysville, alabama 35073 availability ,shift,ays flexible availability customer service representative we want you to join our team as a customer service representative. if you have the desire to be challenged, work in a fast,aced, fun environment and to grow your career ,look no further. as a customer service representative, you will enjoy medical, vision, dental, & life insurance,hort & long term disability flexible schedules weekly pay full,ime or part,ime large, stable employer fast career opportunities work with fun, motivated people task variety paid comprehensive training 401k with a competitive company match flexible spending,ealth savings accounts tuition reimbursement your key responsibilities you will greet customers, run the register, cashier, make purchase suggestions and sometimes work with our food program. there is never a dull moment as you will be working around the store (inside and out) in many different areas to help maintain our high standards for store appearance and provide fast and friendly service to our customers. provide regular and predicable onsite attendance. you will interact with many customers daily, all while working with a fun, energetic team accomplishing daily tasks around the store! you are good at selling products to customers providing excellent customer care communication and friendly conversation performing at a quick pace while having fun working as part of a team to accomplish daily goals coming up with great ideas to solve problems thinking quickly and offering suggestions great if you have retail and customer service experience sales associate or cashiering experience high school diploma or equivalent motivation to advance in your career! willingness to learn and have fun! physical requirements ability to stand and,r walk for up to

In [0]:
from pyspark.sql.functions import col

top_roles = (
    df_features_clean.groupBy("job_role")
                     .count()
                     .orderBy(col("count").desc())
                     .limit(20)
)

display(top_roles)


job_role,count
customer service representative,10010
sales associate ft,7325
store manager,6314
shift manager,6028
accountant,5899
assistant manager,5729
first year tax professional,5356
sales associate pt,4924
registered nurse,4640
account executive,3085


Databricks visualization. Run in Databricks to view.

In [0]:
save_path = "abfss://lakehouse@projectlinkedinjobs.dfs.core.windows.net/gold/linkedin_features_v1/"

df_features_clean.write.format("delta") \
    .mode("overwrite") \
    .save(save_path)

print("Saved clean features to:", save_path)


Saved clean features to: abfss://lakehouse@projectlinkedinjobs.dfs.core.windows.net/gold/linkedin_features_v1/
